In [ ]:
import pandas as pd
import glob
import os

# ── 1. Load all CSV files from both month folders ──────────────────────────
 # adjust to your local path

# Assumes folder structure:
#   Fitabase Data 3.12.16-4.11.16/
#   Fitabase Data 4.12.16-5.12.16/

month1 = "./mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16"
month2 = "./mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16"

def load_and_tag(folder, label):
    frames = {}
    for path in glob.glob(os.path.join(folder, "*.csv")):
        name = os.path.basename(path).replace(".csv", "")
        df = pd.read_csv(path)
        df["_source"] = label
        frames[name] = df
    return frames

m1 = load_and_tag(month1, "month1")
m2 = load_and_tag(month2, "month2")

In [12]:

m1

{'dailyActivity_merged':              Id ActivityDate  TotalSteps  TotalDistance  TrackerDistance  \
 0    1503960366    3/25/2016       11004       7.110000         7.110000   
 1    1503960366    3/26/2016       17609      11.550000        11.550000   
 2    1503960366    3/27/2016       12736       8.530000         8.530000   
 3    1503960366    3/28/2016       13231       8.930000         8.930000   
 4    1503960366    3/29/2016       12041       7.850000         7.850000   
 ..          ...          ...         ...            ...              ...   
 452  8877689391     4/8/2016       23014      20.389999        20.389999   
 453  8877689391     4/9/2016       16470       8.070000         8.070000   
 454  8877689391    4/10/2016       28497      27.530001        27.530001   
 455  8877689391    4/11/2016       10622       8.060000         8.060000   
 456  8877689391    4/12/2016        2350       1.780000         1.780000   
 
      LoggedActivitiesDistance  VeryActiveDistance

In [13]:
# ── 2. Merge each matching table by stacking rows ─────────────────────────
# Get all unique table names across both months
all_tables = set(m1.keys()) | set(m2.keys())

merged = {}
for table in all_tables:
    parts = []
    if table in m1: parts.append(m1[table])
    if table in m2: parts.append(m2[table])
    merged[table] = pd.concat(parts, ignore_index=True)
    print(f"{table:45s}  rows: {len(merged[table]):>6,}")

minuteStepsWide_merged                         rows: 21,645
minuteCaloriesWide_merged                      rows: 21,645
hourlySteps_merged                             rows: 46,183
minuteIntensitiesNarrow_merged                 rows: 2,770,620
minuteMETsNarrow_merged                        rows: 2,770,620
heartrate_seconds_merged                       rows: 3,638,339
hourlyIntensities_merged                       rows: 46,183
minuteStepsNarrow_merged                       rows: 2,770,620
weightLogInfo_merged                           rows:    100
dailySteps_merged                              rows:    940
minuteIntensitiesWide_merged                   rows: 21,645
dailyIntensities_merged                        rows:    940
sleepDay_merged                                rows:    413
dailyActivity_merged                           rows:  1,397
minuteCaloriesNarrow_merged                    rows: 2,770,620
dailyCalories_merged                           rows:    940
minuteSleep_merged       

In [14]:

# ── 3. Pull out the key tables ────────────────────────────────────────────
daily    = merged.get("dailyActivity_merged")
sleep    = merged.get("sleepDay_merged")
hourly_i = merged.get("hourlyIntensities_merged")
hourly_s = merged.get("hourlySteps_merged")
hourly_c = merged.get("hourlyCalories_merged")
hr       = merged.get("heartrate_seconds_merged")

In [15]:

# ── 4. Parse & normalize timestamps ──────────────────────────────────────
# Daily activity
daily["date"] = pd.to_datetime(daily["ActivityDate"], format="%m/%d/%Y")
daily.drop(columns=["ActivityDate"], inplace=True)

In [16]:

# Sleep (timestamp includes time — strip to date)
sleep["date"] = pd.to_datetime(sleep["SleepDay"], format="%m/%d/%Y %I:%M:%S %p").dt.normalize()
sleep.drop(columns=["SleepDay"], inplace=True)

In [17]:

# Hourly tables
for df in [hourly_i, hourly_s, hourly_c]:
    df["datetime"] = pd.to_datetime(df["ActivityHour"], format="%m/%d/%Y %I:%M:%S %p")
    df["date"]     = df["datetime"].dt.normalize()
    df["hour"]     = df["datetime"].dt.hour
    df.drop(columns=["ActivityHour"], inplace=True)

# Heart rate
hr["datetime"] = pd.to_datetime(hr["Time"], format="%m/%d/%Y %I:%M:%S %p")
hr.drop(columns=["Time"], inplace=True)

In [18]:

# ── 5. Deduplicate ────────────────────────────────────────────────────────
daily = daily.drop_duplicates(subset=["Id", "date"])
sleep = sleep.drop_duplicates(subset=["Id", "date"])

# ── 6. Drop device-off rows (0 steps AND 0 calories) ─────────────────────
before = len(daily)
daily = daily[~((daily["TotalSteps"] == 0) & (daily["Calories"] == 0))]
print(f"\nDropped {before - len(daily)} device-off rows from daily table")

# ── 7. Enrich daily table ─────────────────────────────────────────────────
daily["day_of_week"]  = daily["date"].dt.day_name()
daily["is_weekend"]   = daily["date"].dt.dayofweek >= 5
daily["week_number"]  = daily["date"].dt.isocalendar().week.astype(int)

# ── 8. Merge daily + sleep ────────────────────────────────────────────────
sleep_cols = ["Id", "date", "TotalSleepRecords", "TotalMinutesAsleep", "TotalTimeInBed"]
daily_sleep = daily.merge(sleep[sleep_cols], on=["Id", "date"], how="left")

# Sleep efficiency (NaN where no sleep data)
daily_sleep["sleep_efficiency"] = (
    daily_sleep["TotalMinutesAsleep"] / daily_sleep["TotalTimeInBed"]
).round(3)

# ── 9. Merge hourly tables into one wide hourly frame ─────────────────────
hourly = hourly_i.merge(
    hourly_s[["Id", "datetime", "StepTotal"]],
    on=["Id", "datetime"], how="left"
).merge(
    hourly_c[["Id", "datetime", "Calories"]],
    on=["Id", "datetime"], how="left"
)
hourly["day_of_week"] = hourly["datetime"].dt.day_name()
hourly["is_weekend"]  = hourly["datetime"].dt.dayofweek >= 5

# ── 10. Quick sanity check ────────────────────────────────────────────────
print("\n── daily_sleep ──")
print(f"  Shape         : {daily_sleep.shape}")
print(f"  Date range    : {daily_sleep['date'].min().date()} → {daily_sleep['date'].max().date()}")
print(f"  Unique users  : {daily_sleep['Id'].nunique()}")
print(f"  Sleep coverage: {daily_sleep['TotalMinutesAsleep'].notna().mean():.1%} of days")

print("\n── hourly ──")
print(f"  Shape         : {hourly.shape}")
print(f"  Unique users  : {hourly['Id'].nunique()}")
sdfasdfasdf
print(f"  Shape         : {hr.shape}")
print(f"  Unique users  : {hr['Id'].nunique()}")



Dropped 9 device-off rows from daily table

── daily_sleep ──
  Shape         : (1364, 23)
  Date range    : 2016-03-12 → 2016-05-12
  Unique users  : 35
  Sleep coverage: 30.0% of days

── hourly ──
  Shape         : (47233, 11)
  Unique users  : 35

── heart rate ──
  Shape         : (3638339, 4)
  Unique users  : 15


In [2]:
daily.shape

NameError: name 'daily' is not defined